# A/B Test Analysis Workflow
This notebook performs an end-to-end A/B test analysis including:
* Data validation & Quality checks (SRM test)
* Conversion Rate (CR) analysis (Z-test / Chi-Square)
* Revenue / ARPU analysis (t-test, Mann-Whitney U, Bootstrap)
* Decision making & Visualizations

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.stats.api as sms
import seaborn as sns
import matplotlib.pyplot as plt

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data & Initial Inspection

In [ ]:
# Sample size parameters for the FitLife project
n_a, n_b = 24100, 24350
price = 9.99

np.random.seed(42)

# Group A (Control)
converted_a = np.random.binomial(1, 0.0150, n_a)
df_a = pd.DataFrame({
    'user_id': range(1, n_a + 1),
    'group': 'A',
    'converted': converted_a,
    'revenue': converted_a * price
})

# Group B (Variant)
converted_b = np.random.binomial(1, 0.0187, n_b)
df_b = pd.DataFrame({
    'user_id': range(n_a + 1, n_a + n_b + 1),
    'group': 'B',
    'converted': converted_b,
    'revenue': converted_b * price
})

df = pd.concat([df_a, df_b], ignore_index=True)
df.head()

## 2. Sample Ratio Mismatch (SRM) Check
Check if the traffic split between variants matches the expected ratio (e.g., 50/50).

In [ ]:
observed = df['group'].value_counts()
total_users = observed.sum()
expected = [total_users / 2, total_users / 2]  # Assuming 50/50 split

chi2_stat, p_val_srm = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"SRM Check P-Value: {p_val_srm:.4f}")

if p_val_srm < 0.01:
    print("WARNING: Sample Ratio Mismatch detected! Investigate traffic assignment.")
else:
    print("SUCCESS: No SRM detected. Traffic split is balanced.")

## 3. Conversion Rate (CR) Analysis
Compare conversion rates using a two-proportion Z-test and Chi-Square test.

In [ ]:
summary_cr = df.groupby('group')['converted'].agg(['count', 'sum', 'mean']).rename(columns={'mean': 'conversion_rate'})
summary_cr['conversion_rate_pct'] = summary_cr['conversion_rate'] * 100
print(summary_cr)

# Proportions Z-Test
count_conv = summary_cr['sum']
n_obs = summary_cr['count']
z_stat, cr_p_val = sms.proportions_ztest(count_conv, n_obs)

cr_a = summary_cr.loc['A', 'conversion_rate_pct']
cr_b = summary_cr.loc['B', 'conversion_rate_pct']
relative_lift = ((cr_b - cr_a) / cr_a) * 100

print(f"\n--- Conversion Analysis Result ---")
print(f"Relative Lift: {relative_lift:.2f}%")
print(f"Z-Test P-Value: {cr_p_val:.4f}")

if cr_p_val < 0.05:
    print("Statistically Significant Result: Reject the Null Hypothesis.")
else:
    print("Not Statistically Significant: Fail to Reject the Null Hypothesis.")

## 4. Revenue & ARPU Analysis
Evaluate Average Revenue Per User (ARPU) using T-Test and Bootstrap.

In [ ]:
# 1. Filter revenue series for both groups
rev_a = df[df['group'] == 'A']['revenue']
rev_b = df[df['group'] == 'B']['revenue']

arpu_a = rev_a.mean()
arpu_b = rev_b.mean()
arpu_lift_pct = ((arpu_b - arpu_a) / arpu_a) * 100

print(f"ARPU Group A: ${arpu_a:.4f}")
print(f"ARPU Group B: ${arpu_b:.4f}")
print(f"Observed ARPU Difference: ${arpu_b - arpu_a:.4f} ({arpu_lift_pct:.2f}% lift)\n")

# 2. Calculate and print ARPPU
arppu = df[df['converted'] == 1].groupby('group')['revenue'].mean()
print("--- ARPPU (Paying Users Only) ---")
print(f"ARPPU Group A: ${arppu.loc['A']:.2f}")
print(f"ARPPU Group B: ${arppu.loc['B']:.2f}")

# 3. Bootstrap procedure for ARPU difference
np.random.seed(42)
n_iterations = 3000
boot_diffs = []

for _ in range(n_iterations):
    boot_a = np.random.choice(rev_a, size=len(rev_a), replace=True)
    boot_b = np.random.choice(rev_b, size=len(rev_b), replace=True)
    boot_diffs.append(boot_b.mean() - boot_a.mean())

# 4. Calculate 95% Confidence Interval
ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])

print(f"\n--- ARPU Bootstrap Analysis Result ---")
print(f"95% CI for ARPU Difference: [${ci_lower:.4f}, ${ci_upper:.4f}]")

if ci_lower > 0:
    print("Statistically Significant Result: 95% CI does not include 0.")
else:
    print("Not Statistically Significant: 95% CI includes 0.")

## 5. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style settings
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 6))

categories = ['Control (A)', 'Variant (B)']
conversion_rates = [1.53, 1.87]
# Set symmetric errors for CI (e.g., +- 0.15%)
ci_errors = [0.15, 0.16] 

bars = ax.bar(
    categories, 
    conversion_rates, 
    yerr=ci_errors, 
    capsize=8, 
    color=['#808B96', '#2ECC71'], 
    edgecolor='black', 
    linewidth=1
)

# Add value labels above the bars
for bar, rate in zip(bars, conversion_rates):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0, 
        height + 0.2, 
        f'{rate:.2f}%', 
        ha='center', 
        va='bottom', 
        fontsize=12, 
        fontweight='bold'
    )

ax.set_ylim(0, 2.5)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Paywall A/B Test: Conversion Rate Comparison', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', visible=False) # Disable vertical gridlines

plt.tight_layout()
plt.savefig('paywall_ab_test_fixed.png', dpi=300)
plt.show()